In [14]:
import json
import re

def all_fields_match(metadata, fields):
    """
    takes a dictionary containing the mapped metadata in metasra and 
    a dictionary of field names to filter as key and corresponding filter terms
    as regular expressions and returns True if all fields match the filter term
    else False
    
    :param metadata:   dictionary containing the mapped metadata from metasra
    :param fields:     dictonary with fields to filter as key and values to filter as compiled re objects (see re package)
    
    :return:           True if all fields match else False
    """
    matching_results = []
    for field, matcher in fields.items():
        fieldvalue = metadata[field]
        match = matcher.match(fieldvalue)
        matching_results.append(True if match else False)
    
    return all(matching_results)
    

def filter_metasra_by_fields(db, fields):
    """
    takes a the loaded metasra db and a dictionary with fields to filter as keys and 
    field values to filter by as corresponding values.
    
    Returns the filtered db
    
    :param db:        metasra db dictionary returned by json.load
    :param fields:    dictionary with fields to filter as keys and values to filter by as values
    
    :return:          filtered db
    """
    
    fields = {
        field: re.compile(filtervalue) for field, filtervalue in fields.items()
    }
    
    filtered_db = dict()
    for sraid, mapped_metadata in db.items():
        if all_fields_match(mapped_metadata, fields):
            filtered_db[sraid] = mapped_metadata
        
    return filtered_db
    

fields = {
    'assay': 'RNA-seq',
    'species': 'human'
}


# metasra db is indexed by SRA sample accessions start with SRS
with open('metasra.v1-8.json', 'r') as metasradb:
    db = json.load(metasradb)
    filtered_db = filter_metasra_by_fields(
        db, 
        fields
    )

len(filtered_db)

343431

In [19]:
import urllib3
import certifi

http = urllib3.PoolManager(
    ca_certs = certifi.where(),
    cert_reqs = 'CERT_NONE'                      
)

# retrieving number of studies that match search criteria
r = http.request(
    'GET', 
    'http://metasra.biostat.wisc.edu/api/v01/samples.json',
    fields = {
        'study': 'SRS3181591',
        'limit': 1,
        'skip': 0
    }
)

json.loads(r.data.decode('utf-8'))

/Users/dmalzl/miniconda3/envs/scllm/lib/python3.12/site-packages/urllib3/connectionpool.py:1100: InsecureRequestWarning: Unverified HTTPS request is being made to host 'metasra.biostat.wisc.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'skip': 0,
 'sampleCount': 0,
 'limit': 1,
 'studies': [],
 'studyCount': 0,
 'terms': []}